In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

SILVER_TABLE = "supply_chain_dev.shipments.silver_shipment_events"
CURRENT_STATE = "supply_chain_dev.shipments.current_shipment_state"
GOLD_TABLE = "supply_chain_dev.shipments.gold_shipment_summary"

print("Config ready!")
print(f"Source 1: {SILVER_TABLE}")
print(f"Source 2: {CURRENT_STATE}")
print(f"Target  : {GOLD_TABLE}")

Config ready!
Source 1: supply_chain_dev.shipments.silver_shipment_events
Source 2: supply_chain_dev.shipments.current_shipment_state
Target  : supply_chain_dev.shipments.gold_shipment_summary


In [0]:
# Gold layer — carrier performance summary
# This is what a logistics manager would look at every morning

gold_df = spark.table(CURRENT_STATE) \
    .groupBy("carrier", "status") \
    .agg(
        count("booking_id").alias("shipment_count"),
        round(avg("weight_kg"), 2).alias("avg_weight_kg"),
        countDistinct("origin_port").alias("unique_origins"),
        countDistinct("destination_port").alias("unique_destinations")
    ) \
    .orderBy("carrier", "shipment_count".desc() if False else desc("shipment_count"))

gold_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(GOLD_TABLE)

print(f"Gold table created!")
print(f"Total rows: {spark.table(GOLD_TABLE).count()}")
display(spark.table(GOLD_TABLE))


Gold table created!
Total rows: 27


carrier,status,shipment_count,avg_weight_kg,unique_origins,unique_destinations
CMA CGM,CUSTOMS_HOLD,6,9106.44,4,4
CMA CGM,IN_TRANSIT,4,12252.63,3,4
CMA CGM,OUT_FOR_DELIVERY,3,17399.47,3,2
CMA CGM,PORT_ARRIVED,3,12371.68,3,2
CMA CGM,VESSEL_DEPARTED,1,11296.85,1,1
COSCO,OUT_FOR_DELIVERY,6,17336.38,6,5
COSCO,IN_TRANSIT,4,5357.89,4,1
COSCO,CUSTOMS_HOLD,3,12262.25,3,3
COSCO,CUSTOMS_RELEASED,3,14289.17,3,3
COSCO,PORT_ARRIVED,2,9525.69,2,2


In [0]:
display(spark.table(GOLD_TABLE).orderBy("carrier", desc("shipment_count")))

carrier,status,shipment_count,avg_weight_kg,unique_origins,unique_destinations
CMA CGM,CUSTOMS_HOLD,6,9106.44,4,4
CMA CGM,IN_TRANSIT,4,12252.63,3,4
CMA CGM,PORT_ARRIVED,3,12371.68,3,2
CMA CGM,OUT_FOR_DELIVERY,3,17399.47,3,2
CMA CGM,VESSEL_DEPARTED,1,11296.85,1,1
COSCO,OUT_FOR_DELIVERY,6,17336.38,6,5
COSCO,IN_TRANSIT,4,5357.89,4,1
COSCO,CUSTOMS_HOLD,3,12262.25,3,3
COSCO,CUSTOMS_RELEASED,3,14289.17,3,3
COSCO,PORT_ARRIVED,2,9525.69,2,2


In [0]:

# Time travel — see Gold table history
GOLD_TABLE = "supply_chain_dev.shipments.gold_shipment_summary"

display(spark.sql(f"DESCRIBE HISTORY {GOLD_TABLE}"))


version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
0,2026-06-23T22:20:08.000Z,8479271986133632,ravigroverus@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.format.version"":""2.12.0"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(3305384530794884),d9699c17-990f-4ca8-884d-2fb0819bcda5,0623-204258-8vs7ytvj-v2n,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 27, numOutputBytes -> 2622)",null,Databricks-Runtime/18.2.x-aarch64-photon-scala2.13


In [0]:
from pyspark.sql.functions import col, desc

GOLD_TABLE = "supply_chain_dev.shipments.gold_shipment_summary"
SILVER_TABLE = "supply_chain_dev.shipments.silver_shipment_events"

# Time travel — read Silver at Version 0 (before our MERGE)
silver_v0 = spark.read.format("delta") \
    .option("versionAsOf", 0) \
    .table(SILVER_TABLE)

# Time travel — read Silver at Version 1 (after our MERGE)
silver_v1 = spark.read.format("delta") \
    .option("versionAsOf", 1) \
    .table(SILVER_TABLE)

print("=== Time Travel Audit ===")
print(f"Silver Version 0 — total records : {silver_v0.count()}")
print(f"Silver Version 1 — total records : {silver_v1.count()}")
print(f"Silver Version 0 — DELIVERED count: {silver_v0.filter(col('status') == 'DELIVERED').count()}")
print(f"Silver Version 1 — DELIVERED count: {silver_v1.filter(col('status') == 'DELIVERED').count()}")
print("\nThis proves we can audit exactly what the data looked like at any point in time")

=== Time Travel Audit ===
Silver Version 0 — total records : 431
Silver Version 1 — total records : 431
Silver Version 0 — DELIVERED count: 0
Silver Version 1 — DELIVERED count: 5

This proves we can audit exactly what the data looked like at any point in time
